<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 3: 编码注意力机制

In [71]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.10.0


本章介绍注意力机制，即LLM的引擎:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="900px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="800px">

## 3.1 长序列建模中的问题

由于源语言和目标语言的语法结构存在差异，逐字翻译文本是不可行的：:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="900px">

- 在Transformer模型出现之前，编码器-解码器循环神经网络（RNN）常用于机器翻译任务。
- 编码器将源语言的一串词元序列作为输入，并通过隐藏状态（一个中间神经网络层）编码整个输入序列的压缩表示。然后，解码器利用其当前的隐藏状态开始逐个词元进行翻译:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="900px">

编码器-解码器RNN的一个主要限制是，在解码阶段，RNN无法直接访问编码器中的早期隐藏状态。因此，它只能依赖当前的隐藏状态，这个状态包含了所有相关信息。这可能导致上下文丢失，特别是在复杂句子中，依赖关系可能跨越较长的距离。

## 3.2 使用注意力机制捕捉数据依赖关系

- 通过注意力机制，网络中的文本生成解码器部分能够选择性地访问所有输入词元，这意味着在生成特定输出词元的过程中，某些输入词元比其他词元更重要:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="900px">

- Transformer 模型中的自注意力机制是一种旨在增强输入表示的技术，它使序列中的每个位置都能与其他位置互动，并确定彼此的相关性

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="900px">

## 3.3 通过自注意力机制关注输入的不同部分

### 3.3.1 没有可训练权重的简单自注意力机制

- 本节讲解一种**极度简化版的自注意力机制**，其中**不含任何可训练参数**
- 本节内容**仅用于演示说明**，**并非**Transformer中实际使用的注意力机制
- 下一节（3.3.2节）将在这种简单注意力机制的基础上进行扩展，实现**真正的自注意力机制**
- 假设给定输入序列  $x^{(1)}$  到  $x^{(T)}$ 
    - 输入为一段文本（例如“Your journey starts with one step”这样的句子），且已按照第2章所述转换为**词嵌入（token embeddings）**
    - 例如， $x^{(1)}$  是一个d维向量，代表单词“Your”，以此类推
- **目标**：为输入序列中的每个元素  $x^{(i)}$ （从  $x^{(1)}$  到  $x^{(T)}$ ）计算上下文向量 $z^{(i)}$ （其中  $z$  与  $x$  维度相同）
    - 上下文向量  $z^{(i)}$  是对输入  $x^{(1)}$  到  $x^{(T)}$  的加权和
    - 上下文向量是针对特定输入的“上下文”表示
        - 不用  $x^{(i)}$  表示任意输入词，我们以第二个输入  $x^{(2)}$  为例
        - 继续用具体示例说明：不用占位符  $z^{(i)}$ ，而是看第二个输出上下文向量  $z^{(2)}$ 
        - 第二个上下文向量  $z^{(2)}$ ，是以第二个输入元素 $x^{(2)}$ 为参照，对所有输入  $x^{(1)}$  到  $x^{(T)}$  进行加权求和
        - **注意力权重**决定了在计算  $z^{(2)}$  时，每个输入元素对加权和的贡献大小
        - 简而言之，可以把  $z^{(2)}$  看作是  $x^{(2)}$  的**改进版本**：它在保留自身信息的同时，还融入了与当前任务相关的**所有其他输入元素的信息**

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="900px">

- (请注意，为了减少视觉混乱，图中数字已截断至小数点后一位；同样，其他数字也可能包含截断值。)

- 按照惯例，未归一化的注意力权重被称为“注意力得分”，而归一化的注意力得分（总和为 1）被称为“注意力权重”。图中$\omega$是分数，$\alpha$是权重

- 以下代码将逐步执行上图所示的操作。

- **Step 1:** 计算非标准化注意力分数 $\omega$
- 假设我们使用第二个输入token作为查询，即, $q^{(2)} = x^{(2)}$, 我们通过点积计算未归一化的注意力得分:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- 以上$\omega$ 是希腊字母 "omega" 用于表示未归一化的注意力得分
    - 下标 "21" $\omega_{21}$ 这意味着输入序列元素 2 被用作针对输入序列元素 1 的查询序列

假设我们有如下输入句子，它已经按照第 3 章所述嵌入到 3 维向量中（为了便于说明，我们在这里使用了一个非常小的嵌入维度，以便它可以完整地显示在页面上而无需换行）。:

In [72]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- （本书遵循机器学习和深度学习的常用约定，将训练样本表示为行，特征值表示为列；以上述张量为例，每一行代表一个词，每一列代表一个嵌入维度。）

- 本节的主要目标是演示上下文向量如何运作 $z^{(2)}$ 使用第二个输入序列进行计算, $x^{(2)}$, 作为查询

- 该图描绘了此过程的初始步骤，即计算注意力得分 ω $x^{(2)}$ 以及所有其他输入元素，通过点积运算进行运算。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="900px">

🎯 我们以输入序列元素 $x^{(2)}$ 为例，计算上下文向量 $z^{(2)}$。 在本节稍后部分，我们将推广此方法以计算所有上下文向量

**1️⃣ 第一步** 是通过计算查询词 $x^{(2)}$ 与所有其他输入词元之间的点积来计算未归一化的注意力分数

In [73]:
# 第二个输入词元作为查询向量
query = inputs[1]

# 创建一个长度为 inputs.shape[0] （即6）的空张量，用于存储注意力分数。
# 这里 inputs.shape[0] 返回输入张量的第一个维度大小，即输入序列的长度（6个词）。
attn_scores_2 = torch.empty(inputs.shape[0])

# i为index, x_i是具体的值
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


> 补充说明：点积本质上是将两个向量逐元素相乘，然后将结果相加的简写形式:

In [74]:
# 演示手动算点积和dot算点积
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


**2️⃣ 第二步:** 将未归一化的注意力得分 ("omegas", $\omega$) 进行归一化，以便它们的总和为1（这是一种约定俗成的做法，有利于解释，并且对训练稳定性很重要）:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="900px">

In [75]:
# 手动执行归一化
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


- 然而，在实践中，使用softmax函数进行归一化是常见的做法，也是推荐的做法。softmax函数更擅长处理极端值，并且在训练过程中具有更理想的梯度特性。.

In [76]:
# 这里自己简单实现了一个softmax函数进行缩放，它还会对向量元素进行归一化，使它们的总和为1
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


> ## 为什么简单归一化和softmax归一化结果不同？
> 两种方法产生不同结果的原因在于它们处理注意力分数的方式不同：
> 
> 1. 简单归一化 ：只是线性地缩放所有分数，保持它们之间的相对比例不变。
> 2. Softmax归一化 ： 
>       - 通过指数函数放大了较大分数的权重
>       - 压缩了较小分数的权重
>       - 产生了更"尖锐"的概率分布
>       - 使得较大的注意力分数获得相对更多的权重
> 
> 从结果可以看出，Softmax归一化后：
>   - 较大的分数（如第二个和第三个元素）获得了更多权重
>   - 较小的分数获得了更少权重

⚠️ 上述简单实现方法对于过大或过小的输入值都可能由于溢出和下溢问题而出现数值不稳定问题。   
💡 因此，在实践中，建议使用 PyTorch 实现的 softmax 函数，该函数针对性能进行了高度优化:    

In [77]:
# 使用 PyTorch 实现的 softmax 函数
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


**3️⃣ 第三步**: 通过将嵌入的输入token与注意力权重相乘再求和计算得到上下文向量 $z^{(2)}$ 

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

In [78]:
query = inputs[1] # 2nd input token is the query
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i
print(context_vec_2)

# 上面的等价写法
context_vec_2 = inputs.T @ attn_weights_2
print(context_vec_2)
print(inputs.shape)
print(inputs.T.shape)
print(attn_weights_2.shape)

tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])
torch.Size([6, 3])
torch.Size([3, 6])
torch.Size([6])


### 3.3.2 计算所有输入词元的注意力权重

#### 推广到所有输入序列标记:
🎯 上面我们计算了输入 2 的注意力权重和上下文向量（如下图中高亮显示的行所示），接下来我们将这种计算方法推广到计算所有注意力权重和上下文向量。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="400px">

> 请注意，为了减少视觉混乱，图中数字已截断至小数点后两位；每行数值之和应为 1.0 或 100%；类似地，其他图中的数字也已截断

在自我注意过程中，首先计算注意力得分，然后对这些得分进行归一化，得出总和为 1 的注意力权重。   
然后利用这些注意力权重，通过对输入进行加权求和来生成上下文向量。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="400px">

In [79]:
# 计算所有的注意力得分
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [80]:
# 可以通过矩阵乘法更高效地实现上述目标
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [81]:
# 将得分执行归一化得到注意力权重
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [82]:
# 抽查一下第二行的和是否为1
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

# 应该每一行中的值之和都为1才对（dim=-1意思是按行算）
print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [83]:
# 计算全部的注意力上下文向量
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


从结果可以看到，第二行 $z^{(2)} = [0.4419, 0.6515, 0.5683]$ 可以和前文计算的对得上，证明推广到所有的算法没问题: 

In [84]:
print("之前计算的第二个上下文向量:", context_vec_2)

之前计算的第二个上下文向量: tensor([0.4419, 0.6515, 0.5683])


## 3.4 实现带可训练权重的自注意力机制

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="900px">

### 3.4.1 逐步计算注意力权重

- 在本节中，我们将实现原始Transformer架构、GPT模型以及大多数其他流行的LLM模型中使用的自注意力机制
- 这种自注意力机制也称为“缩放点积注意力”
- 总体思路与之前类似:
  - 我们希望计算上下文向量，将其作为特定于某个输入元素的输入向量的加权和
  - 对于上述内容，我们需要注意力权重
- 正如你所看到的，它与前面介绍的基本注意力机制相比，只有细微的差别:
  - 最显著的区别在于引入了权重矩阵，这些权重矩阵会在模型训练过程中进行更新
  - 这些可训练的权重矩阵至关重要，它们能帮助模型（特别是模型内部的注意力模块）学习生成“好的”上下文向量

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="800px">

我们将逐步实现自注意力机制，首先介绍三个训练权重矩阵 $W_q$, $W_k$, $W_v$
- 这三个矩阵用于投影嵌入的输入token, $x^{(i)}$, 通过矩阵乘法将其转换为**查询**向量、**键**向量和**值**向量:

  - 查询向量: $q^{(i)} = x^{(i)}\,W_q $
  - 键向量: $k^{(i)} = x^{(i)}\,W_k $
  - 值向量: $v^{(i)} = x^{(i)}\,W_v $


> 输入向量$x$和查询向量$q$的嵌入维度可以相同，也可以不同，这取决于模型的设计和具体实现   
> 在GPT模型中，输入和输出维度通常相同，但为了便于说明，更好地展示计算过程，我们在此选择不同的输入和输出维度:

In [85]:
x_2 = inputs[1] # 第二个输入元素
d_in = inputs.shape[1] # 输入嵌入大小，d=3
d_out = 2 # 输出嵌入大小，d=2

> 下面，我们初始化三个权重矩阵；请注意，为了便于说明，我们将 requires_grad 设置为 False，以减少输出中的冗余信息。但如果要将这些权重矩阵用于模型训练，则需要将 requires_grad 设置为 True，以便在模型训练期间更新这些矩阵。

In [86]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [87]:
# 接下来，我们计算查询向量、键向量和值向量：
query_2 = x_2 @ W_query # _2 因为它与第二个输入元素有关。
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


In [88]:
# 如下所示，我们已成功将 6 个输入标记从 3D 空间投影到 2D 嵌入空间
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="900px">

In [89]:
# 下一步，我们通过计算查询向量与每个键向量之间的点积来计算未归一化的注意力分数:
keys_2 = keys[1] # Python starts index at 0
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)


In [90]:
# 由于我们有6个输入，因此对于给定的查询向量，我们有6个注意力分数:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="900px">

In [91]:
# 接下来我们使用之前用过的 softmax 函数计算注意力权重（归一化的注意力得分之和为 1）
# 与之前不同的是，现在我们将注意力得分除以嵌入维度的平方根来进行缩放, d_k**0.5表示平方根
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


> ### 为什么进行这样的缩放   
>    - 防止梯度消失 ：当嵌入维度$d_k$较大时，点积结果的数值会变得很大，导致 softmax 函数的输出趋近于 one-hot 向量，使得梯度消失，影响模型训练。   
>    - 保持方差稳定 ：根据数学推导，点积的结果的方差会随着$d_k$的增加而线性增长。通过除以$\sqrt{d_k}$ ，可以保持点积结果的方差稳定在1左右，避免数值不稳定问题。   
>    - 改善训练稳定性 ：稳定的方差有助于模型在训练过程中更快收敛，减少训练过程中的波动。   
>    - 理论基础 ：这一缩放操作是 Transformer 论文中提出的标准做法，被称为"缩放点积注意力"（Scaled Dot-Product Attention）。   

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="900px">

In [92]:
# 现在我们计算输入查询向量 2 的上下文向量，就是用权重点积值矩阵:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### 3.4.2 实现一个简化的自注意Python类

In [93]:
# 综上所述，我们可以按如下方式实现自注意力机制
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="900px">

In [94]:
# 我们可以使用 PyTorch 的线性层来简化上述实现，如果禁用偏置单元，线性层就等价于矩阵乘法。
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # 使用 `nn.Linear` 相比我们手动使用 `nn.Parameter(torch.rand(...)` 方法的另一个巨大优势在于，`nn.Linear` 具有更优的权重初始化方案，这使得模型训练更加稳定。
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


> 请注意，SelfAttention_v1 和 SelfAttention_v2 会给出不同的输出，因为它们对权重矩阵使用了不同的初始权重

## 3.5 利用因果注意力隐藏未来词汇

在因果注意力机制中，对角线以上的注意力权重会被屏蔽，从而确保对于任何给定的输入，LLM 在利用注意力权重计算上下文向量时，无法利用未来的词元。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="600px">

### 3.5.1 因果注意力的掩码实现

因果自注意力机制确保模型对序列中特定位置的预测仅依赖于先前位置的已知输出，而不依赖于未来位置的输出。简单来说，这确保了每个后续单词的预测都只依赖于前面的单词。为了实现这一点，对于每个给定的token，我们屏蔽掉后面的token

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="900px">

In [95]:
# 重用查询矩阵和键权重矩阵
# 为了方便起见，我们使用上一节中的 SelfAttention_v2 对象。
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [96]:
# 屏蔽未来注意力权重的最简单方法是使用 PyTorch 的 tril 函数创建一个掩码，将主对角线下方的元素（包括对角线本身）设置为 1，将主对角线上方的元素设置为 0。
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [97]:
# 然后，我们可以将注意力权重与该掩码相乘，从而将对角线以上的注意力分数归零
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


- 但是，如果像上面那样在 softmax 之后应用掩码，就会破坏 softmax 生成的概率分布。
- Softmax 确保所有输出值之和为 1
- 在softmax之后进行掩蔽操作需要重新归一化输出，使其总和再次为1，这会使过程复杂化，并可能导致意想不到的效果

In [98]:
# 为了确保各行之和为 1，我们可以按如下方式归一化注意力权重
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


💡 虽然我们现在已经完成了因果注意力机制的编码工作，但让我们简要地看一下实现上述相同功能的更高效方法：与其将对角线以上的注意力权重置零并重新归一化结果，我们可以在未归一化的注意力得分进入 softmax 函数之前，用负无穷大掩盖对角线以上的未归一化注意力得分:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="600px">

In [99]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
attn_scores_masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(attn_scores_masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [100]:
# 如下所示，现在每一行的注意力权重之和又正确地为1了
attn_weights = torch.softmax(attn_scores_masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 利用dropout掩码额外的注意力权重

- 此外，我们还应用 dropout 来减少训练过程中的过拟合
- Dropout 可以应用于多个领域:
  - 例如，在计算注意力权重之后
  - 或者将注意力权重与值向量相乘后
- 这里，我们将在计算注意力权重之后应用dropout掩码，因为这样做更常见

> 此外，在这个具体例子中，我们使用了 50% 的 dropout 率，这意味着随机丢弃一半的注意力权重。（之后训练 GPT 模型时，我们会使用更低的 dropout 率，例如 0.1 或 0.2）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="400px">

- 如果我们采用 0.5 (50%) 的dropout率，则未dropout的值将按 1/0.5 = 2 的比例进行相应缩放。
- 缩放比例由以下公式计算得出 1 / (1 - `dropout_rate`)

In [101]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
example = torch.ones(6, 6) # create a matrix of ones

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [102]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


> 请注意，根据操作系统不同，最终的输出结果可能有所不同；您可以阅读更多关于这种[不一致性的信息](https://github.com/pytorch/pytorch/issues/121595)

### 3.5.3 实现一个简化的因果注意力类

💬 现在，我们已准备好实现一个可用的自注意力机制，包括因果掩码和dropout掩码   
🎯 还有一点需要实现，即编写代码来处理包含多个输入的批次，以便我们的 `CausalAttention` 类能够支持我们在第二章中实现的数据加载器生成的批次输出

In [106]:
# 为简单起见，为了模拟这种批量输入，我们复制输入文本示例
print(inputs)
batch = torch.stack((inputs, inputs), dim=0)
print(batch)
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
torch.Size([2, 6, 3])


In [115]:
# 一个简化的因果注意力类
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # 新增
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # 新增

    def forward(self, x):
        # 新增的批量纬度b；num_tokens代表整个词集合有多少个词；d_in代表输入token的纬度
        b, num_tokens, d_in = x.shape # ❓ b，d_in在后面都没有使用，我理解是继承的nn.Module完成的批量处理？
        # 如果输入数量 `num_tokens` 超过 `context_length`，则会导致下面的mask创建部分错误
        # 实际上，这并不是问题，因为LLM（第4-7章）确保了输入 
        # 在到达此forward方法之前，不要超过 `context_length`
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2) # 将维度1和2转置，将批维度保持在第一个位置（0）
        attn_scores.masked_fill_(  # 新增, 在PyTorch中，带有尾随的下划线的操作将就地执行，从而避免了不必要的内存副本
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights) # 新增，在归一化得到权重后执行dropout操作，作用是避免过拟合

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print(context_vecs)
print("两个批次，每个批次6个单词，每个单词2个维度")
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
两个批次，每个批次6个单词，每个单词2个维度
context_vecs.shape: torch.Size([2, 6, 2])


> ⚠️ 请注意，dropout 仅在训练期间应用，推理期间不应用

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="900px">

## 3.6 将单头注意力扩展到多头注意力

&nbsp;
### 3.6.1 叠加多个单头注意力层

以下是之前实现的自注意力机制的概要，这单个的因果注意力也被称为单头注意力:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="700px">

🎯 我们只需将多个单头注意力模块堆叠起来，即可得到一个多头注意力模块:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="700px">

💬 多头注意力机制的核心思想是使用不同的、已学习的线性投影多次（并行）运行注意力机制。这使得模型能够联合关注来自不同位置的不同表征子空间的信息。

In [117]:
# 一个实现多头注意力的封装类
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # 创建一个 nn.ModuleList ，包含 num_heads 个 CausalAttention 实例
        # 每个 CausalAttention 实例使用相同的参数配置
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        # 对输入 x 并行应用所有注意力头
        # 将每个注意力头的输出在最后一个维度（特征维度）上拼接
        # 最终输出的维度为 (batch_size, sequence_length, d_out * num_heads)
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


💬 在上述实现中，嵌入维度为 4，因为我们将键向量、查询向量、值向量以及上下文向量的嵌入维度都设置为 2。由于我们有两个注意力头，所以输出嵌入维度为 2*2=4。

### 3.6.2 通过权重划分实现多头注意力

以上是对多头注意力机制的一种直观且功能齐全的实现（封装了之前提到的单头注意力机制 CausalAttention）   

🎯 但我们也可以编写一个名为 MultiHeadAttention 的独立类来实现相同的功能   

In [137]:
# 在这个独立的 MultiHeadAttention 类中，我们不会将单个注意力头连接起来
# 相反，我们会创建单独的 W_query、W_key 和 W_value 权重矩阵，然后将它们拆分成每个注意力头的单独矩阵
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 减少投影维度以匹配所需的输出维度；（//在python中是整除的意思，无论是否能整除都返回整数）
        print(self.head_dim)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 使用一个线形层来组合头的输出
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        # 批次，每句6个单词，每个单词3个维度嵌入
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 我们通过添加 `num_heads` 维度来隐式地分割矩阵 
        # 展开最后一个维度: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 转置: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 使用因果掩码计算缩放点积注意力（又称自注意力）
        attn_scores = queries @ keys.transpose(2, 3)  # 每个头的点积

        # 原始掩码被截断为标记数并转换为布尔值
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 使用掩码来填充注意力分数
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 形状: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # 合并头部，其中 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # 添加一个可选的线性投影

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 4
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

2
tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


> 请注意，以上内容本质上是 `MultiHeadAttentionWrapper` 的重写版本，效率更高。
> 由于随机权重初始化不同，最终输出结果看起来略有差异，但两者都是功能齐全的实现，可以用于我们将在后续章节中实现的 GPT 类。

> **关于输出维度的说明**
>   - 在上面的 MultiHeadAttention 类中，我使用了 d_out=2，以与之前 MultiHeadAttentionWrapper 类中的设置保持一致。
>   - 由于使用了连接操作，MultiHeadAttentionWrapper 返回的输出头维度为 d_out * num_heads（即 2 * 2 = 4）。
>   - 然而，MultiHeadAttention 类（为了更方便用户使用）允许我们直接通过 d_out 来控制输出头维度；这意味着，如果我们设置 d_out = 2，则输出头维度将为 2，而与头的数量无关。
>   - 事后看来，正如读者[指出](https://github.com/rasbt/LLMs-from-scratch/pull/859)的那样，使用 d_out = 4 的 MultiHeadAttention 可能更直观，这样它就能产生与 d_out = 2 的MultiHeadAttentionWrapper 相同的输出维度。

> ⚠️ 请注意，我们还在上面的 `MultiHeadAttention` 类中添加了一个线性投影层（`self.out_proj`）。这只是一个简单的线性变换，不会改变维度。在 LLM 实现中使用这样的投影层是一种标准惯例，但并非绝对必要（最近的研究表明，移除该投影层不会影响建模性能；请参阅本章末尾的延伸阅读部分）。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="700px">

> 请注意，如果您对上述功能的紧凑高效实现感兴趣，也可以考虑使用 PyTorch 中的 [torch.nn.MultiheadAttention](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 类。

- 由于上述实现乍一看可能有点复杂，让我们来看看执行 `attn_scores = queries @ keys.transpose(2, 3)` 时会发生什么:

In [37]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- 在这种情况下，PyTorch 中的矩阵乘法实现会处理 4 维输入张量，因此矩阵乘法会在最后两个维度（num_tokens 和 head_dim）之间进行，然后对每个头部重复此操作。
- 例如，以下方法可以更简洁地分别计算每个头部的矩阵乘法：

In [38]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


&nbsp;
## Summary and takeaways

- See the [./multihead-attention.ipynb](./multihead-attention.ipynb) code notebook, which is a concise version of the data loader (chapter 2) plus the multi-head attention class that we implemented in this chapter and will need for training the GPT model in upcoming chapters
- You can find the exercise solutions in [./exercise-solutions.ipynb](./exercise-solutions.ipynb)